# Backtesting Framework Tutorial

This notebook demonstrates how to use the **Event-Driven Backtesting Framework** to:

1. Download and prepare historical market data
2. Run a backtest with the Moving Average Crossover strategy
3. Run a backtest with the Mean Reversion (Bollinger Bands) strategy
4. Analyse performance metrics
5. Generate a visual tear sheet
6. Optimise strategy parameters with Grid Search

---

## 0. Setup

Make sure you have installed all dependencies:

```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

## 1. Download Historical Data

We use the built-in `download_data.py` script to fetch data from Yahoo Finance and save it as Parquet files.

In [ ]:
from data.download_data import download_and_clean

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# Download AAPL data (2018-2024)
download_and_clean(
    symbol='AAPL',
    start='2018-01-01',
    end='2024-12-31',
    output_dir=DATA_DIR,
)

print('\nData files:')
for f in os.listdir(DATA_DIR):
    if f.endswith('.parquet'):
        print(f'  {f}')

## 2. Inspect the Data

In [ ]:
import pandas as pd

df = pd.read_parquet(os.path.join(DATA_DIR, 'AAPL.parquet'))
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
df.head()

## 3. Backtest: Moving Average Crossover

This trend-following strategy goes **LONG** when the 50-day MA crosses above the 200-day MA, and **EXITs** on the reverse crossover.

In [ ]:
from backtester import BacktestEngine, HistoricParquetDataHandler
from strategies.ma_crossover import MovingAverageCrossoverStrategy

# Create data handler
data_handler = HistoricParquetDataHandler(
    data_dir=DATA_DIR,
    symbol_list=['AAPL'],
)

# Create strategy
strategy = MovingAverageCrossoverStrategy(
    data_handler=data_handler,
    symbol='AAPL',
    short_window=50,
    long_window=200,
)

# Create and run the engine
engine = BacktestEngine(
    data_handler=data_handler,
    strategy=strategy,
    portfolio_kwargs={'initial_capital': 100_000, 'order_quantity': 100},
    execution_kwargs={
        'commission_type': 'fixed',
        'commission_value': 1.0,
        'slippage_type': 'spread_pct',
        'slippage_value': 0.05,
    },
)
engine.run()

print('Backtest complete!')

### 3.1 Performance Summary

In [ ]:
summary = engine.get_performance_summary()
for k, v in summary.items():
    print(f'{k:30s}: {v:>12.4f}')

### 3.2 Equity Curve

In [ ]:
import matplotlib.pyplot as plt

ec = engine.get_equity_curve()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Equity curve
axes[0].plot(ec.index, ec['total'], color='#6366f1', linewidth=1.5)
axes[0].fill_between(ec.index, ec['total'], alpha=0.15, color='#6366f1')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('MA Crossover (50/200) — AAPL')
axes[0].grid(True, alpha=0.3)

# Drawdown
hwm = ec['total'].cummax()
dd = (ec['total'] - hwm) / hwm * 100
axes[1].fill_between(dd.index, dd.values, 0, color='#ef4444', alpha=0.4)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Trade Log

In [ ]:
trades = engine.get_trade_log()
print(f'Total fills: {len(trades)}')
trades.head(10)

### 3.4 Generate Tear Sheet

In [ ]:
tearsheet_path = engine.generate_tearsheet(
    output_path=os.path.join(PROJECT_ROOT, 'results', 'ma_crossover_tearsheet.html'),
    strategy_name='MA Crossover (50/200) — AAPL',
)
print(f'Tear sheet saved to: {tearsheet_path}')

## 4. Backtest: Mean Reversion (Bollinger Bands)

This strategy goes **LONG** when the price falls below the lower Bollinger Band and **EXITs** when price exceeds the upper band.

In [ ]:
from strategies.mean_reversion import MeanReversionBollingerStrategy

# Fresh data handler
data_handler_2 = HistoricParquetDataHandler(
    data_dir=DATA_DIR,
    symbol_list=['AAPL'],
)

strategy_2 = MeanReversionBollingerStrategy(
    data_handler=data_handler_2,
    symbol='AAPL',
    lookback=20,
    num_std=2.0,
)

engine_2 = BacktestEngine(
    data_handler=data_handler_2,
    strategy=strategy_2,
    portfolio_kwargs={'initial_capital': 100_000, 'order_quantity': 100},
    execution_kwargs={
        'commission_type': 'fixed',
        'commission_value': 1.0,
        'slippage_type': 'spread_pct',
        'slippage_value': 0.05,
    },
)
engine_2.run()

summary_2 = engine_2.get_performance_summary()
for k, v in summary_2.items():
    print(f'{k:30s}: {v:>12.4f}')

In [ ]:
tearsheet_path_2 = engine_2.generate_tearsheet(
    output_path=os.path.join(PROJECT_ROOT, 'results', 'mean_reversion_tearsheet.html'),
    strategy_name='Mean Reversion (BB 20/2.0) — AAPL',
)
print(f'Tear sheet saved to: {tearsheet_path_2}')

## 5. Strategy Comparison

In [ ]:
comparison = pd.DataFrame({
    'MA Crossover': summary,
    'Mean Reversion': summary_2,
}).T

comparison.style.format('{:.4f}').set_caption('Strategy Comparison')

## 6. Parameter Optimisation (Grid Search)

We optimise the MA Crossover parameters using **Grid Search** with a 70/30 In-Sample / Out-of-Sample split to prevent overfitting.

In [ ]:
from backtester.optimizer import GridSearchOptimizer

def make_data_handler(start_date=None, end_date=None):
    return HistoricParquetDataHandler(
        data_dir=DATA_DIR,
        symbol_list=['AAPL'],
        start_date=start_date,
        end_date=end_date,
    )

optimizer = GridSearchOptimizer(
    strategy_cls=MovingAverageCrossoverStrategy,
    param_grid={
        'short_window': [20, 50, 100],
        'long_window': [100, 150, 200],
    },
    data_handler_factory=make_data_handler,
    portfolio_kwargs={'initial_capital': 100_000, 'order_quantity': 100},
    execution_kwargs={'commission_type': 'fixed', 'commission_value': 1.0},
    optimise_metric='Sharpe Ratio',
    oos_fraction=0.3,
)

opt_results = optimizer.run()
print('Top 5 parameter combinations (by In-Sample Sharpe Ratio):')
opt_results.head()

### 6.1 IS vs OOS Performance (Overfitting Check)

In [ ]:
if 'IS_Sharpe Ratio' in opt_results.columns and 'OOS_Sharpe Ratio' in opt_results.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(opt_results))
    ax.bar([i - 0.15 for i in x], opt_results['IS_Sharpe Ratio'], width=0.3,
           label='In-Sample', color='#6366f1', alpha=0.8)
    ax.bar([i + 0.15 for i in x], opt_results['OOS_Sharpe Ratio'], width=0.3,
           label='Out-of-Sample', color='#22c55e', alpha=0.8)
    ax.set_xlabel('Parameter Combination')
    ax.set_ylabel('Sharpe Ratio')
    ax.set_title('In-Sample vs Out-of-Sample Sharpe Ratio')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Optimisation columns not found. Run the grid search above first.')

## 7. Writing Your Own Strategy

To create a custom strategy, inherit from the `Strategy` base class and implement `calculate_signals()`:

```python
from backtester.strategy import Strategy
from backtester.events import MarketEvent, SignalEvent, SignalDirection

class MyCustomStrategy(Strategy):
    def __init__(self, data_handler, symbol='AAPL', **params):
        self.data_handler = data_handler
        self.symbol = symbol
        # Store your parameters ...
        self._in_position = False

    def calculate_signals(self, event):
        # Access latest data
        prices = self.data_handler.get_latest_bars_values(
            self.symbol, 'Adj Close', n=20
        )
        if len(prices) < 20:
            return None

        # Your logic here ...
        if should_buy and not self._in_position:
            self._in_position = True
            return SignalEvent(
                symbol=self.symbol,
                datetime=self.data_handler.get_latest_bar_datetime(self.symbol),
                direction=SignalDirection.LONG,
            )
        return None
```

---

**End of Tutorial** — For more details, see the `README.md` and the docstrings in each module.